# ProstT5 Speculative Decoding — Folding Direction (AA → 3Di)

This notebook benchmarks **speculative decoding** for ProstT5 *folding* using the enc-CNN as a drafter.

**Direction:** AA → 3Di (opposite of the inverse-folding notebooks)
- Input: amino acid sequence (uppercase, space-separated, prefixed with `<AA2fold>`)
- Output: 3Di structural-alphabet sequence (20 lowercase letters, same alphabet as AAs)

**Drafter:** Tiny CNN (234k params) that reads ProstT5 encoder hidden states and predicts all 3Di positions in a single parallel pass.

**Integration:** HuggingFace `assistant_model` API — no custom speculative decoding loop.

**Structure:**
- **Setup** — installs, imports, model loading, dataset
- **CNNAssistantModel** — HF-compatible wrapper for the folding CNN
- **Sanity check** — verify HF assisted output matches plain greedy output
- **Benchmark** — sweep K ∈ {1, 3, 5, 8} across 100 proteins
- **α analysis** — theoretical acceptance rate + Theorem 3.8 prediction
- **Results** — tables and plots

In [1]:
%pip install -q tiktoken sentencepiece
%pip install -q 'accelerate>=0.26.0'
%pip install -q "transformers==4.46.3" "huggingface_hub>=0.23.2,<1.0" sentencepiece
%pip install -q pandas matplotlib tqdm

In [ ]:
import os, time, statistics, gc
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from tqdm.auto import tqdm
import pandas as pd
import matplotlib.pyplot as plt
plt.rcParams.update({"figure.dpi": 120, "font.size": 10})

from transformers import T5Tokenizer, AutoModelForSeq2SeqLM, GenerationConfig, PreTrainedModel
from transformers.modeling_outputs import ModelOutput   # dict-like base class; safe across versions

try:
    from google.colab import drive
    drive.mount('/content/drive')
    USE_DRIVE = True
    DRIVE_ROOT = '/content/drive/MyDrive'
except Exception:
    USE_DRIVE = False
    DRIVE_ROOT = None

if torch.cuda.is_available():
    device = torch.device("cuda:0")
elif torch.backends.mps.is_available():
    device = torch.device("mps")
else:
    device = torch.device("cpu")

print(f"PyTorch: {torch.__version__}  |  Device: {device}")
if device.type == "cpu":
    print("WARNING: running on CPU — timings will not reflect realistic GPU latency.")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
PyTorch: 2.11.0+cu128  |  Device: cuda:0


In [3]:
# @title [Optional] GPU Memory Cleanup — run before re-running the notebook
_to_clear = ['model', 'encoder', 'cnn', 'cnn_assistant']
for _v in _to_clear:
    if _v in globals():
        del globals()[_v]

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    free, total = torch.cuda.mem_get_info()
    print(f"GPU memory cleared.  Free: {free/1e9:.1f} GB / {total/1e9:.1f} GB total")
else:
    print("No CUDA GPU — nothing to clear.")

GPU memory cleared.  Free: 15.5 GB / 15.6 GB total


## 1. Configuration & Model Loading

In [ ]:
# @title 1A. Configuration
PROSTT5_NAME = "Rostlab/ProstT5_fp16"
NOTEBOOK_DIR = Path(".").resolve()

# 3Di alphabet — same 20 letters as AAs but lowercase
DI_LETTERS = "acdefghiklmnpqrstvwy"
assert len(DI_LETTERS) == 20

# Benchmark protocol
NUM_WARMUP  = 1   # warmup passes before timing
NUM_REPEATS = 3   # timed repeats; report median

# Plot colours — covers every integer K from 1..16 via a continuous colormap
import matplotlib.cm as _cm
_pal_cmap = _cm.get_cmap("tab10")
PALETTE = {k: _pal_cmap(i % 10) for i, k in enumerate(range(1, 17))}
BLUE   = "#2e86de"
ORANGE = "#f39c12"
GREEN  = "#27ae60"
GRAY   = "#7f8c8d"

# ── Locate folding CNN checkpoint (AA → 3Di) ──────────────────────────────
# Path.exists() can return True on Google Drive before the file is downloaded
# (lazy mount). We verify the file is actually readable instead.
def _readable(path: Path) -> bool:
    try:
        with open(path, "rb") as f:
            f.read(4)
        return True
    except OSError:
        return False

CNN_CKPT = None
for candidate in [
    NOTEBOOK_DIR / "model.pt",
    NOTEBOOK_DIR / "CNN_fromAA_to3Di.pt",
    NOTEBOOK_DIR / "cnn_chkpnt_folding" / "model.pt",
    NOTEBOOK_DIR.parent / "cnn_chkpnt_folding" / "model.pt",
    *([] if not USE_DRIVE else [
        Path(DRIVE_ROOT) / "cnn_chkpnt_folding" / "model.pt",
        Path(DRIVE_ROOT) / "models" / "CNN_fromAA_to3Di.pt",
        Path(DRIVE_ROOT) / "model.pt",
    ]),
]:
    if _readable(candidate):
        CNN_CKPT = candidate
        break

if CNN_CKPT is None:
    raise FileNotFoundError(
        "Folding CNN checkpoint not found or not readable.\n"
        "Upload cnn_chkpnt_folding/model.pt to the notebook directory or Google Drive."
    )
print(f"Folding CNN checkpoint: {CNN_CKPT}")

In [6]:
# @title 1B. Load ProstT5 + Folding CNN

# ── Tokenizer ────────────────────────────────────────────────────────────
if 'tokenizer' not in globals():
    print("Loading tokenizer...")
    tokenizer = T5Tokenizer.from_pretrained(PROSTT5_NAME, do_lower_case=False, legacy=True)
else:
    print("Tokenizer already in memory.")

# ── ProstT5 encoder-decoder (~5.6 GB) ────────────────────────────────────
if 'model' not in globals():
    print("Loading ProstT5 model (~5.6 GB)...")
    dtype = torch.float16 if device.type == "cuda" else torch.float32
    model = AutoModelForSeq2SeqLM.from_pretrained(
        PROSTT5_NAME, low_cpu_mem_usage=True, torch_dtype=dtype,
    ).to(device).eval()
    print(f"ProstT5: {sum(p.numel() for p in model.parameters())/1e6:.1f}M params")
else:
    print("ProstT5 already in memory.")
encoder = model.get_encoder()

# ── Folding CNN (AA → 3Di, same Conv2d architecture as the AA-CNN) ───────
class FoldingCNN(nn.Module):
    """CNN head: encoder hidden states (1024-dim) → 3Di logits (20 classes)."""
    def __init__(self, num_tokens: int = 20, hidden: int = 32, in_dim: int = 1024):
        super().__init__()
        self.classifier = nn.Sequential(
            nn.Conv2d(in_dim, hidden, kernel_size=(7, 1), padding=(3, 0)),
            nn.ReLU(),
            nn.Dropout(0.0),
            nn.Conv2d(hidden, num_tokens, kernel_size=(7, 1), padding=(3, 0)),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x.permute(0, 2, 1).unsqueeze(-1)   # (B, 1024, L, 1)
        x = self.classifier(x)                  # (B, 20, L, 1)
        return x.squeeze(-1).permute(0, 2, 1)   # (B, L, 20)

if 'cnn' not in globals():
    cnn = FoldingCNN().to(device).eval()
    ckpt = torch.load(CNN_CKPT, map_location=device, weights_only=False)
    cnn.load_state_dict(ckpt.get("state_dict", ckpt), strict=True)
    print(f"Folding CNN loaded: {sum(p.numel() for p in cnn.parameters()):,} params")
else:
    print("CNN already in memory.")

# ── Token ID mappings ─────────────────────────────────────────────────────
# ProstT5 generates space-separated tokens; 3Di tokens are lowercase
# (▁a, ▁c, ...) — must encode with a leading space, not bare letter.
DI_TOKEN_IDS           = [tokenizer.encode(f" {di}", add_special_tokens=False)[0] for di in DI_LETTERS]
CNN_IDX_TO_DI_TOKEN_ID = {i: tid for i, tid in enumerate(DI_TOKEN_IDS)}
DI_TOKEN_ID_TO_CNN_IDX = {tid: i for i, tid in enumerate(DI_TOKEN_IDS)}
DECODER_START_TOKEN_ID = model.config.decoder_start_token_id

# Sanity-check token round-trip
for di, tid in zip(DI_LETTERS[:3], DI_TOKEN_IDS[:3]):
    decoded = tokenizer.decode([tid]).strip()
    assert decoded == di, f"Token mismatch: '{di}' → {tid} → '{decoded}'"

print(f"Decoder start token ID : {DECODER_START_TOKEN_ID}")
print(f"3Di token IDs (first 5): {list(zip(DI_LETTERS[:5], DI_TOKEN_IDS[:5]))}")
if torch.cuda.is_available():
    used  = torch.cuda.memory_allocated() / 1e9
    total = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU memory: {used:.1f} GB used / {total:.1f} GB total")

Loading tokenizer...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Loading ProstT5 model (~5.6 GB)...
ProstT5: 2818.9M params
Folding CNN loaded: 233,908 params
Decoder start token ID : 0
3Di token IDs (first 5): [('a', 128), ('c', 147), ('d', 135), ('e', 134), ('f', 140)]
GPU memory: 7.3 GB used / 15.6 GB total


## 2. Dataset — 100-Protein Test Set

In [ ]:
# @title 2A. Load FASTA files (generated by prostT5_baseline_performance.ipynb)
def parse_fasta(path: Path) -> dict:
    seqs, cur = {}, None
    for line in path.read_text(encoding="utf-8").splitlines():
        if line.startswith(">"):
            cur = line[1:].split()[0]; seqs[cur] = ""
        elif cur:
            seqs[cur] += line.strip()
    return seqs

_drive_candidates = [] if not USE_DRIVE else [
    Path(DRIVE_ROOT) / "models",
    Path(DRIVE_ROOT) / "benchmark_data",
    Path(DRIVE_ROOT) / "prostT5" / "benchmark_data",
    Path(DRIVE_ROOT) / "Speculative-Decoding-ProstT5" / "prostT5" / "benchmark_data",
    Path(DRIVE_ROOT) / "prostT5_benchmarks",
]
DATA_DIR = None
for candidate in [
    NOTEBOOK_DIR / "benchmark_data",
    Path("/content/benchmark_data"),
    Path("/content/prostT5/benchmark_data"),
    *_drive_candidates,
]:
    if (candidate / "test_set_AA.fasta").exists():
        DATA_DIR = candidate
        break

if DATA_DIR is None:
    raise FileNotFoundError(
        "test_set_AA.fasta not found.\n"
        "Run prostT5_baseline_performance.ipynb first to generate the FASTA files."
    )

aa_seqs = parse_fasta(DATA_DIR / "test_set_AA.fasta")
di_seqs = parse_fasta(DATA_DIR / "test_set_3Di.fasta")

# Build benchmark list — AA is input, 3Di is ground truth for folding direction
benchmark_data = [
    {"id": uid, "AA": aa_seqs[uid], "3Di": di_seqs[uid].lower()}
    for uid in aa_seqs if uid in di_seqs
]
lengths = [len(r["AA"]) for r in benchmark_data]

print(f"Loaded {len(benchmark_data)} proteins from {DATA_DIR}")
print(f"Length range: {min(lengths)}–{max(lengths)} AA  |  median: {sorted(lengths)[len(lengths)//2]}")
print(f"First 3: {[(r['id'], len(r['AA'])) for r in benchmark_data[:3]]}")

Loaded 100 proteins from /content/drive/MyDrive/models
Length range: 46–2554 AA  |  median: 419
First 3: [('P01308', 110), ('P02798', 61), ('P62944', 937)]


In [ ]:
# @title 2B. Hyperparameters — edit before running

# Draft lengths to sweep
K_VALUES     = [1, 2, 4]

# Set to an int (e.g. 10) for a quick test, None for all 100 proteins
NUM_PROTEINS = 50

if NUM_PROTEINS is not None:
    benchmark_data = benchmark_data[:NUM_PROTEINS]
    print(f"Dataset limited to {NUM_PROTEINS} proteins.")
else:
    print(f"Using all {len(benchmark_data)} proteins.")
print(f"K_VALUES={K_VALUES}  |  NUM_WARMUP={NUM_WARMUP}  |  NUM_REPEATS={NUM_REPEATS}")

Dataset limited to 50 proteins.
K_VALUES=[1, 2, 4]  |  NUM_WARMUP=1  |  NUM_REPEATS=3


## 3. CNNAssistantModel — HuggingFace-Compatible Wrapper

Wraps the folding CNN as a `PreTrainedModel` so we can pass it to `model.generate(..., assistant_model=cnn_assistant)`.

**Key design:**
- The CNN is *prefix-independent*: it computes all L position logits in one shot from the encoder hidden states, regardless of what was previously generated.
- Logits are cached after the first call per protein (identified by `id(encoder_outputs[0])`), so the CNN runs only once per protein despite multiple decoder steps.
- `attention_mask` is accepted in both `prepare_inputs_for_generation` and `forward` to satisfy HuggingFace's kwargs validation.

In [ ]:
# @title 3. CNNAssistantModel wrapper

class CNNAssistantModel(PreTrainedModel):
    """Wraps the folding CNN (AA→3Di) as a HF-compatible assistant for speculative decoding."""
    config_class = type(model.config)

    def __init__(self, config, prostt5_encoder, cnn_head, di_token_ids):
        super().__init__(config)
        self._encoder      = prostt5_encoder
        self._cnn          = cnn_head
        self._di_token_ids = di_token_ids   # list of 20 3Di vocab IDs

        self.config.is_encoder_decoder     = True
        self.config.decoder_start_token_id = config.decoder_start_token_id
        self.generation_config = GenerationConfig(
            num_assistant_tokens=5,
            num_assistant_tokens_schedule="constant",
            do_sample=False,
            max_length=3000,
        )
        self._cached_logits    = None
        self._cached_hidden_id = None

    # ── CNN → full-vocabulary logits ──────────────────────────────────────
    def _full_vocab_logits(self, encoder_outputs) -> torch.Tensor:
        """Run CNN on encoder hidden states; expand to full ProstT5 vocab."""
        hidden   = encoder_outputs[0]        # (1, enc_len, 1024)
        h        = hidden[:, 1:-1, :]        # trim task-prefix token + EOS
        logits20 = self._cnn(h.float())[0]  # (L, 20)
        vocab_size = self.config.vocab_size
        full = torch.full(
            (logits20.shape[0], vocab_size), -1e4,
            device=logits20.device, dtype=logits20.dtype,
        )
        for cnn_idx, tok_id in enumerate(self._di_token_ids):
            full[:, tok_id] = logits20[:, cnn_idx]
        return full   # (L, vocab)

    # ── Required HF hooks ────────────────────────────────────────────────
    def get_encoder(self):
        return self._encoder

    def prepare_inputs_for_generation(
        self, decoder_input_ids,
        encoder_outputs=None,
        attention_mask=None,
        **kwargs,
    ):
        return {
            "decoder_input_ids": decoder_input_ids,
            "encoder_outputs":   encoder_outputs,
            "attention_mask":    attention_mask,
        }

    def forward(
        self,
        decoder_input_ids=None,
        encoder_outputs=None,
        attention_mask=None,
        **kwargs,
    ):
        # Compute (or retrieve cached) CNN logits for this encoder input
        if encoder_outputs is not None:
            h_id = id(encoder_outputs[0])
            if self._cached_hidden_id != h_id:
                self._cached_logits    = self._full_vocab_logits(encoder_outputs)
                self._cached_hidden_id = h_id

        n = decoder_input_ids.shape[1]
        if self._cached_logits is not None:
            L_cached = self._cached_logits.shape[0]
            n_ret    = min(n, L_cached)
            logits   = self._cached_logits[:n_ret].unsqueeze(0)   # (1, n_ret, vocab)
            if n_ret < n:
                pad    = torch.full(
                    (1, n - n_ret, logits.shape[-1]), -1e4,
                    device=logits.device, dtype=logits.dtype,
                )
                logits = torch.cat([logits, pad], dim=1)
        else:
            logits = torch.zeros(
                1, n, self.config.vocab_size, device=device, dtype=torch.float32,
            )
        # ModelOutput is an OrderedDict subclass — supports "key in output",
        # output["logits"], and iteration, all of which HF's generation code relies on.
        return ModelOutput(logits=logits)


# Instantiate
if 'cnn_assistant' not in globals():
    cnn_assistant = CNNAssistantModel(
        config          = model.config,
        prostt5_encoder = encoder,
        cnn_head        = cnn,
        di_token_ids    = DI_TOKEN_IDS,
    ).to(device).eval()

print("CNNAssistantModel created.")
print(f"  is_encoder_decoder   = {cnn_assistant.config.is_encoder_decoder}")
print(f"  num_assistant_tokens = {cnn_assistant.generation_config.num_assistant_tokens}")

CNNAssistantModel created.
  is_encoder_decoder   = True
  num_assistant_tokens = 1


## 4. Helper Functions

In [ ]:
# @title 4. Helper functions

def _sync():
    if device.type == "cuda":  torch.cuda.synchronize()
    elif device.type == "mps": torch.mps.synchronize()


def _format_aa(seq: str) -> str:
    """ProstT5 folding input: <AA2fold> M A L W M R ..."""
    return "<AA2fold> " + " ".join(list(seq.upper()))


def _decode_3di(token_ids) -> str:
    """Token IDs → 3Di string (lowercase, no spaces)."""
    ids = token_ids if isinstance(token_ids, list) else token_ids.tolist()
    s = tokenizer.decode(ids, skip_special_tokens=True)
    return "".join(s.split()).lower()


@torch.inference_mode()
def run_encdec(aa_seq: str) -> tuple[str, float]:
    """Plain enc-dec greedy generation (baseline). Returns (predicted_3Di, median_wall_s)."""
    L   = len(aa_seq)
    enc = tokenizer(
        [_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt"
    ).to(device)
    kw  = dict(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        max_length=L + 2, do_sample=False, num_beams=1,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**kw)
    _sync()
    times = []
    for _ in range(NUM_REPEATS):
        _sync(); t0 = time.perf_counter()
        out = model.generate(**kw)
        _sync(); times.append(time.perf_counter() - t0)
    pred = _decode_3di(out[0])[:L]
    return pred, statistics.median(times)


@torch.inference_mode()
def run_hf_assisted(aa_seq: str, K: int) -> tuple[str, float]:
    """HF assisted generation (speculative decoding). Returns (predicted_3Di, median_wall_s)."""
    L   = len(aa_seq)
    enc = tokenizer(
        [_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt"
    ).to(device)
    cnn_assistant.generation_config.num_assistant_tokens = K
    kw  = dict(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        max_length=L + 2, do_sample=False, num_beams=1,
        assistant_model=cnn_assistant,
    )
    for _ in range(NUM_WARMUP):
        model.generate(**kw)
    _sync()
    times = []
    for _ in range(NUM_REPEATS):
        _sync(); t0 = time.perf_counter()
        out = model.generate(**kw)
        _sync(); times.append(time.perf_counter() - t0)
    pred = _decode_3di(out[0])[:L]
    return pred, statistics.median(times)


print("Helpers defined.")

Helpers defined.


## 5. Sanity Check — HF Output Must Match Plain Greedy Output

Greedy speculative decoding guarantees **identical output** to plain autoregressive greedy generation (Leviathan et al., 2023). We verify this on the 5 shortest proteins before running the full benchmark.

In [ ]:
# @title 5. Sanity check: HF assisted output == plain enc-dec greedy output

print("Verifying correctness on 5 shortest proteins...\n")
test_proteins = sorted(benchmark_data, key=lambda r: len(r["AA"]))[:5]

all_ok = True
for item in test_proteins:
    uid, aa_seq = item["id"], item["AA"]
    L = len(aa_seq)
    ref_3di, _ = run_encdec(aa_seq)

    for K in K_VALUES:
        try:
            pred_3di, _ = run_hf_assisted(aa_seq, K)
            match  = (pred_3di == ref_3di)
            status = "PASS" if match else "FAIL"
            if not match:
                all_ok = False
        except Exception as exc:
            status = f"ERROR: {exc}"
            all_ok = False
        print(f"  {status}  {uid} L={L}  K={K}")

print(f"\n{'ALL CORRECT ✓' if all_ok else 'FAILURES DETECTED — check CNNAssistantModel'}")
if not all_ok:
    print("NOTE: If HF errors out, output_matches will be False in the benchmark.")
    print("      Timing results are still valid; only correctness is affected.")

Verifying correctness on 5 shortest proteins...

  PASS  P01542 L=46  K=1
  PASS  P01542 L=46  K=2
  PASS  P01542 L=46  K=4
  PASS  P0A7N4 L=57  K=1
  PASS  P0A7N4 L=57  K=2
  PASS  P0A7N4 L=57  K=4
  PASS  P02798 L=61  K=1
  PASS  P02798 L=61  K=2
  PASS  P02798 L=61  K=4
  PASS  P24311 L=80  K=1
  PASS  P24311 L=80  K=2
  PASS  P24311 L=80  K=4
  PASS  P0A7U3 L=92  K=1
  PASS  P0A7U3 L=92  K=2
  PASS  P0A7U3 L=92  K=4

ALL CORRECT ✓


## 6. Full Benchmark — enc-dec Baseline + HF Assisted (K sweep)

For each protein:
1. **enc-dec baseline**: plain `model.generate()` with warmup + timed repeats.
2. **HF assisted** for each K: `model.generate(assistant_model=cnn_assistant, num_assistant_tokens=K)`.

Speedup = t_encdec / t_hf_assisted.

In [ ]:
# @title 6. Main benchmark loop

print(f"Benchmarking {len(benchmark_data)} proteins × {len(K_VALUES)} K values")
print(f"Protocol: {NUM_WARMUP} warmup + {NUM_REPEATS} repeats, median wall time\n")

results = []

for item in tqdm(benchmark_data, desc="Proteins"):
    uid, aa_seq = item["id"], item["AA"]
    L = len(aa_seq)

    # ── Baseline ──────────────────────────────────────────────────────────
    pred_dec, t_dec = run_encdec(aa_seq)

    # ── HF assisted for each K ─────────────────────────────────────────────
    for K in K_VALUES:
        try:
            pred_hf, t_hf = run_hf_assisted(aa_seq, K)
            hf_ok = True
        except Exception as e:
            print(f"  {uid} K={K}: HF failed — {e}")
            pred_hf, t_hf, hf_ok = "", float("nan"), False

        results.append(dict(
            id             = uid,
            length         = L,
            K              = K,
            t_encdec       = t_dec,
            t_hf           = t_hf,
            speedup        = t_dec / t_hf if (hf_ok and t_hf > 0) else float("nan"),
            hf_ok          = hf_ok,
            output_matches = (pred_hf == pred_dec) if hf_ok else False,
            pred_3di_dec   = pred_dec,
            pred_3di_hf    = pred_hf,
            gt_3di         = item["3Di"],
        ))

    gc.collect()
    if device.type == "cuda":
        torch.cuda.empty_cache()

df = pd.DataFrame(results)
print(f"\nBenchmark complete. Total rows: {len(df)}")

# Save results — local + Drive
import shutil as _shutil
for _fname, _data in [("folding_spec_dec_results.csv", df)]:
    _local = Path(f"/content/{_fname}")
    _data.to_csv(_local, index=False)
    if USE_DRIVE:
        _shutil.copy(_local, Path(DRIVE_ROOT) / _fname)
        print(f"Saved {_fname} → /content/  and  {DRIVE_ROOT}/")
    else:
        print(f"Saved {_fname} → /content/")

## 7. Acceptance Rate α — Theoretical Analysis

α is the per-position probability that the CNN drafter agrees with the T5 verifier under the sampling distribution:

$$\alpha = \mathbb{E}_x \left[ \sum_t \min(p(t|x), q(t|x)) \right]$$

where `p` is the T5 decoder distribution and `q` is the CNN distribution (both over the 20 3Di tokens).

From Leviathan et al. Theorem 3.8, the expected tokens generated per speculative decoding step with draft length K is:

$$\mathbb{E}[\text{tokens/step}] = \frac{1 - \alpha^{K+1}}{1 - \alpha}$$

This gives a predicted speedup ceiling assuming zero overhead for the drafter.

In [ ]:
# @title 7. Compute α for all proteins (greedy / temperature=1.0)

@torch.inference_mode()
def compute_alpha(aa_seq: str) -> tuple[float, float]:
    """Returns (alpha_mean, alpha_std) for greedy generation."""
    L   = len(aa_seq)
    enc = tokenizer(
        [_format_aa(aa_seq)], add_special_tokens=True, return_tensors="pt"
    ).to(device)

    # Greedy generate (the sequence the verifier produces token-by-token)
    greedy_out = model.generate(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        max_length=L + 2, do_sample=False, num_beams=1,
    )

    # Teacher-forced forward pass → verifier logits at all positions at once
    dec_ids = greedy_out[:, :-1]  # decoder input (shift right of output)
    fwd     = model(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
        decoder_input_ids=dec_ids,
    )
    p_logits = fwd.logits[0, :L]   # (L, vocab)

    # CNN forward → drafter logits
    enc_h    = encoder(
        input_ids=enc.input_ids, attention_mask=enc.attention_mask,
    ).last_hidden_state
    h_cnn    = enc_h[:, 1:-1, :]   # trim prefix + EOS
    q_logits = cnn(h_cnn.float())[0]  # (L_cnn, 20)

    L_eff = min(L, p_logits.shape[0], q_logits.shape[0])

    # Convert to token-level probabilities
    p_probs    = torch.softmax(p_logits[:L_eff].float(), dim=-1)     # (L_eff, vocab)
    q_probs_20 = torch.softmax(q_logits[:L_eff].float(), dim=-1)     # (L_eff, 20)

    # Expand q to full vocabulary (other positions stay 0)
    q_probs = torch.zeros_like(p_probs)
    for i, tid in enumerate(DI_TOKEN_IDS):
        q_probs[:, tid] = q_probs_20[:, i]

    # α per residue = total overlap between p and q
    alpha_per_pos = torch.minimum(p_probs, q_probs).sum(dim=-1)  # (L_eff,)
    return float(alpha_per_pos.mean()), float(alpha_per_pos.std())


print("Computing α for all proteins...")
alpha_rows = []

for item in tqdm(benchmark_data, desc="α computation"):
    uid, aa_seq = item["id"], item["AA"]
    try:
        a_mean, a_std = compute_alpha(aa_seq)
    except Exception as e:
        print(f"  {uid}: α failed — {e}")
        a_mean, a_std = float("nan"), float("nan")
    alpha_rows.append({"id": uid, "length": len(aa_seq),
                       "alpha_mean": a_mean, "alpha_std": a_std})

alpha_df = pd.DataFrame(alpha_rows)

# Save — local + Drive
import shutil as _shutil
_local = Path("/content/folding_alpha_results.csv")
alpha_df.to_csv(_local, index=False)
if USE_DRIVE:
    _shutil.copy(_local, Path(DRIVE_ROOT) / "folding_alpha_results.csv")
    print(f"Saved folding_alpha_results.csv → /content/  and  {DRIVE_ROOT}/")
else:
    print("Saved folding_alpha_results.csv → /content/")

print(f"\nα summary (greedy, T=1.0):")
print(f"  mean  = {alpha_df['alpha_mean'].mean():.4f}")
print(f"  std   = {alpha_df['alpha_mean'].std():.4f}")
print(f"  min   = {alpha_df['alpha_mean'].min():.4f}")
print(f"  max   = {alpha_df['alpha_mean'].max():.4f}")
print(f"\nComparison: inverse-folding (3Di→AA) CNN achieves α ≈ 0.29.")
print(f"The folding CNN achieves α ≈ {alpha_df['alpha_mean'].mean():.2f}, "
      f"meaning 3Di structure is more predictable from AA sequence than vice versa.")

## 8. Results — Tables

In [ ]:
# @title 8A. Per-K summary statistics

# Merge α into main results dataframe
df = df.merge(alpha_df[["id", "alpha_mean", "alpha_std"]], on="id", how="left")

# Predicted tokens/step from Theorem 3.8
def pred_tps(alpha, K):
    if np.isnan(alpha) or alpha >= 0.9999:
        return K + 1
    return (1 - alpha ** (K + 1)) / (1 - alpha)

df["pred_tps"] = df.apply(lambda r: pred_tps(r["alpha_mean"], r["K"]), axis=1)

print("=" * 70)
print("BENCHMARK RESULTS — ProstT5 Folding Speculative Decoding (AA → 3Di)")
print("=" * 70)
print(f"Proteins: {df['id'].nunique()}   Device: {device}")
print()
print(f"{'K':>3}  {'α (mean)':>9}  {'Pred tok/step':>14}  "
      f"{'Speedup mean':>13}  {'Speedup med':>12}  {'Match%':>7}  {'HF OK%':>7}")
print("-" * 75)
for K in K_VALUES:
    sub  = df[df["K"] == K]
    a    = sub["alpha_mean"].mean()
    pt   = sub["pred_tps"].mean()
    sp_m = sub["speedup"].mean()
    sp_d = sub["speedup"].median()
    mt   = sub["output_matches"].mean() * 100
    ok   = sub["hf_ok"].mean() * 100
    print(f"{K:3d}  {a:9.4f}  {pt:14.3f}  {sp_m:13.3f}x  {sp_d:12.3f}x  {mt:6.1f}%  {ok:6.1f}%")
print("=" * 70)
print()
print("Notes:")
print("  Pred tok/step = (1 - α^(K+1)) / (1 - α)  [Leviathan et al., Theorem 3.8]")
print("  Match% = fraction of proteins where HF output == plain greedy output")
print("  (Should be 100% for correct greedy speculative decoding)")

BENCHMARK RESULTS — ProstT5 Folding Speculative Decoding (AA → 3Di)
Proteins: 50   Device: cuda:0

  K   α (mean)   Pred tok/step   Speedup mean   Speedup med   Match%   HF OK%
---------------------------------------------------------------------------
  1     0.6882           1.688          1.447x         1.451x    88.0%   100.0%
  2     0.6882           2.175          1.804x         1.828x    94.0%   100.0%
  4     0.6882           2.782          2.234x         2.258x    84.0%   100.0%

Notes:
  Pred tok/step = (1 - α^(K+1)) / (1 - α)  [Leviathan et al., Theorem 3.8]
  Match% = fraction of proteins where HF output == plain greedy output
  (Should be 100% for correct greedy speculative decoding)


In [ ]:
# @title 8B. Predicted vs measured tokens/step

print("=" * 55)
print("Predicted vs Measured Tokens/Step")
print("=" * 55)
mean_alpha_global = alpha_df["alpha_mean"].mean()
print(f"Global mean α = {mean_alpha_global:.4f}\n")

print(f"{'K':>3}  {'Pred tok/step':>14}  {'Speedup (mean)':>15}  {'Ratio':>7}")
print("-" * 45)
for K in K_VALUES:
    sub  = df[df["K"] == K]
    pred = sub["pred_tps"].mean()
    meas = sub["speedup"].mean()
    ratio = meas / pred if pred > 0 else float("nan")
    print(f"{K:3d}  {pred:14.3f}  {meas:15.3f}  {ratio:7.3f}")

print()
print("Ratio < 1.0 : the CNN drafter's prefix-independence hurts acceptance in practice")
print("Ratio ~ 1.0 : α prediction is accurate")
print("Ratio > 1.0 : measured speedup exceeds theoretical prediction")

Predicted vs Measured Tokens/Step
Global mean α = 0.6882

  K   Pred tok/step   Speedup (mean)    Ratio
---------------------------------------------
  1           1.688            1.447    0.857
  2           2.175            1.804    0.830
  4           2.782            2.234    0.803

Ratio < 1.0 : the CNN drafter's prefix-independence hurts acceptance in practice
Ratio ~ 1.0 : α prediction is accurate
Ratio > 1.0 : measured speedup exceeds theoretical prediction


In [ ]:
# @title 8C. Sequence recovery vs ground-truth 3Di

def seq_recovery(pred: str, gt: str) -> float:
    n = min(len(pred), len(gt))
    if n == 0: return 0.0
    return sum(p == g for p, g in zip(pred[:n], gt[:n])) / n

# Compute per-protein recovery for enc-dec and best-K HF
best_K = K_VALUES[df.groupby("K")["speedup"].mean().values.argmax()]
recovery_rows = []
for item in benchmark_data:
    uid, gt = item["id"], item["3Di"]
    sub_dec = df[(df["id"] == uid) & (df["K"] == K_VALUES[0])]
    sub_hf  = df[(df["id"] == uid) & (df["K"] == best_K)]
    if sub_dec.empty: continue
    pred_dec = sub_dec.iloc[0]["pred_3di_dec"]
    pred_hf  = sub_hf.iloc[0]["pred_3di_hf"] if not sub_hf.empty else ""
    recovery_rows.append({
        "id": uid, "length": len(gt),
        "rec_encdec": seq_recovery(pred_dec, gt),
        "rec_hf":     seq_recovery(pred_hf,  gt),
    })

rec_df = pd.DataFrame(recovery_rows)
print(f"Sequence recovery vs ground-truth 3Di (AlphaFold structures)")
print(f"  enc-dec greedy : {rec_df['rec_encdec'].mean():.3f} ± {rec_df['rec_encdec'].std():.3f}")
print(f"  HF assisted K={best_K}: {rec_df['rec_hf'].mean():.3f} ± {rec_df['rec_hf'].std():.3f}")
print("  (Should be identical if Match% = 100%)")

Sequence recovery vs ground-truth 3Di (AlphaFold structures)
  enc-dec greedy : 0.771 ± 0.143
  HF assisted K=4: 0.771 ± 0.142
  (Should be identical if Match% = 100%)


## 9. Plots

In [ ]:
# @title 9A. Main results figure (6-panel)

fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle(
    "ProstT5 Speculative Decoding — Folding Direction (AA → 3Di)\n"
    "CNN Drafter via HuggingFace assistant_model",
    fontsize=13, y=1.01,
)
axes = axes.flatten()

# ── 1. Speedup distribution by K — boxplot + mean marker ─────────────────
ax = axes[0]
valid_K = [K for K in K_VALUES if df[df["K"] == K]["speedup"].notna().any()]
data_sp = [df[df["K"] == K]["speedup"].dropna().values for K in valid_K]
bp = ax.boxplot(
    data_sp, patch_artist=True, widths=0.5,
    medianprops={"color": "black", "lw": 2},
    showmeans=True,
    meanprops={"marker": "D", "markerfacecolor": "white",
               "markeredgecolor": "black", "markersize": 6},
)
for patch, K in zip(bp["boxes"], valid_K):
    patch.set_facecolor(PALETTE.get(K, BLUE))
    patch.set_alpha(0.75)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.5, label="no speedup (1×)")
ax.set_xticklabels([f"K={K}" for K in valid_K])
ax.set_ylabel("Speedup over plain enc-dec")
ax.set_title("Speedup Distribution by K\n(box=IQR, whisker=5–95%, ◆=mean)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# ── 2. α distribution across proteins ───────────────────────────────────
ax = axes[1]
alpha_vals = alpha_df["alpha_mean"].dropna()
ax.hist(alpha_vals, bins=20, color=BLUE, alpha=0.8, edgecolor="white")
_mean_a = alpha_vals.mean()
_std_a  = alpha_vals.std()
ax.axvline(_mean_a, color="red", linestyle="--", lw=2,
           label=f"mean = {_mean_a:.3f}")
ax.axvspan(_mean_a - _std_a, _mean_a + _std_a, alpha=0.12, color="red",
           label=f"±1 std ({_std_a:.3f})")
ax.set_xlabel("Per-protein acceptance rate α")
ax.set_ylabel("Number of proteins")
ax.set_title("Distribution of Acceptance Rate α (Greedy)")
ax.set_xlim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# ── 3. Predicted tok/step vs measured speedup — error bars on measured ───
ax = axes[2]
mean_a   = alpha_df["alpha_mean"].mean()
K_arr    = np.array(K_VALUES)
pred_tps_vals = [
    (1 - mean_a ** (K + 1)) / (1 - mean_a) if mean_a < 0.9999 else K + 1
    for K in K_arr
]
meas_sp     = [df[df["K"] == K]["speedup"].mean() for K in K_arr]
meas_sp_std = [df[df["K"] == K]["speedup"].std()  for K in K_arr]
x = np.arange(len(K_arr))
w = 0.35
ax.bar(x - w/2, pred_tps_vals, w, color=BLUE,   alpha=0.85,
       label="Pred tok/step (Thm 3.8)")
ax.bar(x + w/2, meas_sp,       w, color=ORANGE, alpha=0.85,
       label="Measured speedup (mean ± std)",
       yerr=meas_sp_std, capsize=5,
       error_kw={"elinewidth": 1.5, "ecolor": "black", "capthick": 1.5})
ax.set_xticks(x)
ax.set_xticklabels([f"K={K}" for K in K_arr])
ax.set_ylabel("Tokens/step  |  Speedup")
ax.set_title("Predicted Tok/Step vs Measured Speedup")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3, axis="y")

# ── 4. Wall time vs protein length ───────────────────────────────────────
ax = axes[3]
sub0 = df[df["K"] == K_VALUES[0]]
ax.scatter(sub0["length"], sub0["t_encdec"],
           color=GRAY, alpha=0.6, s=25, label="enc-dec (baseline)", zorder=3)
for K in K_VALUES:
    sub = df[df["K"] == K].dropna(subset=["t_hf"])
    ax.scatter(sub["length"], sub["t_hf"],
               color=PALETTE.get(K, BLUE), alpha=0.5, s=18, label=f"HF K={K}")
ax.set_xlabel("Protein length (AA residues)")
ax.set_ylabel("Wall time (s)")
ax.set_title("Wall Time vs Protein Length")
ax.set_yscale("log")
ax.set_xscale("log")
ax.legend(fontsize=7, ncol=2)
ax.grid(True, alpha=0.3, which="both")

# ── 5. Speedup vs protein length (best K, coloured by α) ─────────────────
ax = axes[4]
best_K = K_VALUES[int(np.nanargmax([df[df["K"] == K]["speedup"].mean() for K in K_VALUES]))]
sub = df[df["K"] == best_K].dropna(subset=["speedup", "alpha_mean"])
_alpha_min = max(0.0, float(sub["alpha_mean"].min()) - 0.05)
_alpha_max = min(1.0, float(sub["alpha_mean"].max()) + 0.05)
sc = ax.scatter(
    sub["length"], sub["speedup"],
    c=sub["alpha_mean"], cmap="RdYlGn",
    s=40, alpha=0.85, edgecolors="none",
    vmin=_alpha_min, vmax=_alpha_max,
)
ax.axhline(1.0, color="red", linestyle="--", alpha=0.4, label="no speedup")
ax.set_xlabel("Protein length (AA residues)")
ax.set_ylabel(f"Speedup (K={best_K})")
ax.set_title(f"Speedup vs Length (K={best_K}, coloured by α)")
ax.legend(fontsize=8)
fig.colorbar(sc, ax=ax, label="α")
ax.grid(True, alpha=0.3)

# ── 6. Output agreement (Match%) — binomial SE error bars ────────────────
ax = axes[5]
match_rates = [df[df["K"] == K]["output_matches"].mean() * 100 for K in K_VALUES]
# Binomial standard error: SE = sqrt(p*(1-p)/n) * 100
match_se = [
    np.sqrt(p/100 * (1 - p/100) / len(df[df["K"] == K])) * 100
    for p, K in zip(match_rates, K_VALUES)
]
bars = ax.bar(
    [f"K={K}" for K in K_VALUES], match_rates,
    color=[PALETTE.get(K, BLUE) for K in K_VALUES], alpha=0.85, edgecolor="white",
    yerr=match_se, capsize=5,
    error_kw={"elinewidth": 1.5, "ecolor": "black", "capthick": 1.5},
)
ax.axhline(100, color="green", linestyle="--", alpha=0.5, label="expected (100%)")
ax.set_ylim(0, 115)
ax.set_ylabel("Output matches plain greedy (%) ± SE")
ax.set_title("HF Output Agreement with enc-dec Greedy")
ax.legend(fontsize=8)
for bar, pct, se in zip(bars, match_rates, match_se):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + se + 1.5,
            f"{pct:.1f}%", ha="center", va="bottom", fontsize=9)
ax.grid(True, alpha=0.3, axis="y")

plt.tight_layout()

import shutil as _shutil
_plot_path = Path("/content/folding_spec_dec_plots.png")
plt.savefig(_plot_path, bbox_inches="tight", dpi=150)
if USE_DRIVE:
    _shutil.copy(_plot_path, Path(DRIVE_ROOT) / "folding_spec_dec_plots.png")
    print(f"Saved: {_plot_path}  →  also copied to {DRIVE_ROOT}/")
else:
    print(f"Saved: {_plot_path}  (download via Files panel in the Colab sidebar)")
plt.show()

In [ ]:
# @title 9B. α vs protein length + speedup saturation curve

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("α Analysis — Folding Direction (AA → 3Di)", fontsize=12)

# ── Left: α per protein vs length — error bars = per-residue α std ───────
ax = axes[0]
ax.errorbar(
    alpha_df["length"], alpha_df["alpha_mean"],
    yerr=alpha_df["alpha_std"],
    fmt="o", color=BLUE, alpha=0.7,
    ecolor="steelblue", elinewidth=0.8, capsize=3, markersize=4,
    label="per-protein α (error bar = residue-level std)",
)
mean_a = alpha_df["alpha_mean"].mean()
ax.axhline(mean_a, color="red", linestyle="--",
           label=f"mean α = {mean_a:.3f}")
ax.set_xlabel("Protein length (AA residues)")
ax.set_ylabel("Acceptance rate α")
ax.set_title("α vs Protein Length\n(error bars = std over residues within protein)")
ax.set_ylim(0, 1)
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

# ── Right: predicted speedup ceiling vs K — shaded band for α ± std ──────
ax = axes[1]
K_range = np.arange(1, 17)
_a_min = alpha_df["alpha_mean"].min()
_a_max = alpha_df["alpha_mean"].max()
alpha_levels = np.linspace(max(0.1, _a_min - 0.05), min(0.99, _a_max + 0.05), 5)
colors_alpha = plt.cm.plasma(np.linspace(0.15, 0.85, len(alpha_levels)))
for a, col in zip(alpha_levels, colors_alpha):
    tps = [(1 - a**(K+1)) / (1 - a) for K in K_range]
    ax.plot(K_range, tps, "o-", color=col, linewidth=1.5, label=f"α = {a:.2f}")

# Actual mean line + ± 1 std shaded band propagated through Theorem 3.8
mean_a = alpha_df["alpha_mean"].mean()
std_a  = alpha_df["alpha_mean"].std()
tps_mean  = [(1 - mean_a**(K+1)) / (1 - mean_a) if mean_a < 0.9999 else K+1 for K in K_range]
a_lo = max(0.01, mean_a - std_a)
a_hi = min(0.999, mean_a + std_a)
tps_lo = [(1 - a_lo**(K+1)) / (1 - a_lo) for K in K_range]
tps_hi = [(1 - a_hi**(K+1)) / (1 - a_hi) for K in K_range]
ax.plot(K_range, tps_mean, "k--", lw=2, label=f"Actual mean α = {mean_a:.3f}")
ax.fill_between(K_range, tps_lo, tps_hi, color="black", alpha=0.12,
                label=f"±1 std (α ∈ [{a_lo:.2f}, {a_hi:.2f}])")

for K in K_VALUES:
    ax.axvline(K, color="gray", linestyle=":", alpha=0.4)
ax.set_xlabel("Draft length K")
ax.set_ylabel("Expected tokens/step")
ax.set_title("Predicted Tok/Step Ceiling by K and α\n(shaded band = ±1 std of observed α)")
ax.legend(fontsize=8)
ax.grid(True, alpha=0.3)

plt.tight_layout()

import shutil as _shutil
_plot_path = Path("/content/folding_alpha_analysis.png")
plt.savefig(_plot_path, bbox_inches="tight", dpi=150)
if USE_DRIVE:
    _shutil.copy(_plot_path, Path(DRIVE_ROOT) / "folding_alpha_analysis.png")
    print(f"Saved: {_plot_path}  →  also copied to {DRIVE_ROOT}/")
else:
    print(f"Saved: {_plot_path}  (download via Files panel in the Colab sidebar)")
plt.show()

In [ ]:
# @title 9C. Per-protein speedup scatter (all K values)

fig, ax = plt.subplots(figsize=(10, 5))
for K in K_VALUES:
    sub = df[df["K"] == K].dropna(subset=["speedup"]).sort_values("length")
    ax.plot(sub["length"], sub["speedup"],
            "o", alpha=0.5, markersize=5,
            color=PALETTE.get(K, BLUE), label=f"K={K}")

ax.axhline(1.0, color="red", linestyle="--", alpha=0.4, label="no speedup (1×)")
ax.set_xlabel("Protein length (AA residues)")
ax.set_ylabel("Speedup over plain enc-dec")
ax.set_title("Per-Protein Speedup — All K Values (Folding Direction)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()

_plot_path = Path("/content/folding_per_protein_speedup.png")
_drive_plot_path = Path(DRIVE_ROOT) / "folding_per_protein_speedup.png" if USE_DRIVE else None
plt.savefig(_plot_path, bbox_inches="tight", dpi=150)
if _drive_plot_path:
    import shutil; shutil.copy(_plot_path, _drive_plot_path)
    print(f"Saved: {_plot_path}  →  also copied to {_drive_plot_path}")
else:
    print(f"Saved: {_plot_path}  (download via Files panel in the Colab sidebar)")
plt.show()

## 10. Final Summary

In [ ]:
# @title 10. Final summary

mean_alpha = alpha_df["alpha_mean"].mean()

print("=" * 68)
print("FINAL SUMMARY — ProstT5 Speculative Decoding, Folding (AA → 3Di)")
print("=" * 68)
print(f"  Proteins evaluated : {df['id'].nunique()}")
print(f"  Device             : {device}")
print(f"  Drafter            : Folding CNN ({sum(p.numel() for p in cnn.parameters()):,} params)")
print(f"  Verifier           : ProstT5 enc-dec ({sum(p.numel() for p in model.parameters())/1e6:.0f}M params)")
print(f"  Mean α (greedy)    : {mean_alpha:.4f}")
print()
print(f"{'Method':<28} {'α':>7} {'Pred tok/step':>14} {'Speedup (mean)':>15} {'Match%':>8}")
print("-" * 75)
for K in K_VALUES:
    sub  = df[df["K"] == K]
    a    = sub["alpha_mean"].mean()
    pt   = sub["pred_tps"].mean()
    sp   = sub["speedup"].mean()
    mt   = sub["output_matches"].mean() * 100
    print(f"{'HF assisted  K='+str(K):<28} {a:7.4f} {pt:14.3f} {sp:15.3f}x {mt:7.1f}%")
print("=" * 68)
print()
best_K = K_VALUES[int(np.nanargmax([df[df["K"]==K]["speedup"].mean() for K in K_VALUES]))]
best_sp = df[df["K"] == best_K]["speedup"].mean()
print(f"Best K: {best_K}  →  mean speedup {best_sp:.2f}×")
print()
print("Interpretation:")
print(f"  - α = {mean_alpha:.3f} means the CNN drafter agrees with the T5 decoder on ~{mean_alpha*100:.0f}% of tokens.")
pred_best = (1 - mean_alpha**(best_K+1)) / (1 - mean_alpha) if mean_alpha < 0.9999 else best_K+1
print(f"  - Theorem 3.8 predicts {pred_best:.2f} tokens/step for K={best_K}; measured speedup is {best_sp:.2f}×.")
print(f"  - Speedup is bounded by 1/(1-α) = {1/(1-mean_alpha):.2f}× regardless of K.")
print(f"  - To improve: use a higher-α drafter (e.g. small T5 decoder, profile HMM).")

FINAL SUMMARY — ProstT5 Speculative Decoding, Folding (AA → 3Di)
  Proteins evaluated : 50
  Device             : cuda:0
  Drafter            : Folding CNN (233,908 params)
  Verifier           : ProstT5 enc-dec (2819M params)
  Mean α (greedy)    : 0.6882

Method                             α  Pred tok/step  Speedup (mean)   Match%
---------------------------------------------------------------------------
HF assisted  K=1              0.6882          1.688           1.447x    88.0%
HF assisted  K=2              0.6882          2.175           1.804x    94.0%
HF assisted  K=4              0.6882          2.782           2.234x    84.0%

Best K: 4  →  mean speedup 2.23×

Interpretation:
  - α = 0.688 means the CNN drafter agrees with the T5 decoder on ~69% of tokens.
  - Theorem 3.8 predicts 2.71 tokens/step for K=4; measured speedup is 2.23×.
  - Speedup is bounded by 1/(1-α) = 3.21× regardless of K.
  - To improve: use a higher-α drafter (e.g. small T5 decoder, profile HMM).
